In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
current_path = "/content/drive/MyDrive/dl-techniques-01"
os.chdir(current_path)
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/dl-techniques-01


In [ ]:
import numpy as np
from pathlib import Path

from sklearn.metrics import accuracy_score, confusion_matrix

import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image_dataset_from_directory

In [ ]:
batch_size = 32
img_width = 220
img_height = 220

data_path = "./datasets/flowers"

train_ds = image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Found 11200 files belonging to 7 classes.
Using 8960 files for training.
Found 11200 files belonging to 7 classes.
Using 2240 files for validation.


In [ ]:
classes = train_ds.class_names

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
model = tf.keras.Sequential([
    layers.Input(shape=(220, 220, 3)),
    layers.Rescaling(1./ 255),

    layers.Conv2D(16, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPool2D(),
    layers.Dropout(0.2),

    layers.Conv2D(32, kernel_size=(5, 5), padding='same', activation='relu'),
    layers.MaxPool2D(),
    layers.Dropout(0.2),

    layers.Conv2D(64, kernel_size=(5, 5), padding='same', activation='relu'),
    layers.MaxPool2D(),
    layers.Dropout(0.3),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(len(classes), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 220, 220, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 220, 220, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 110, 110, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 110, 110, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 110, 110, 32)   │        12,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 55, 55, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 55, 55, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 55, 55, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 27, 27, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 27, 27, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 46656)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     5,972,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,045,351 (23.06 MB)

 Trainable params: 6,045,351 (23.06 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True
)

In [ ]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 1447s 3s/step - accuracy: 0.3588 - loss: 1.6151 - val_accuracy: 0.5741 - val_loss: 1.2732
Epoch 2/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.5710 - loss: 1.1828 - val_accuracy: 0.5964 - val_loss: 1.1282
Epoch 3/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.6453 - loss: 1.0053 - val_accuracy: 0.6603 - val_loss: 0.9725
Epoch 4/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.7030 - loss: 0.8535 - val_accuracy: 0.6929 - val_loss: 0.9004
Epoch 5/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.7609 - loss: 0.6878 - val_accuracy: 0.7196 - val_loss: 0.8094
Epoch 6/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.8213 - loss: 0.5180 - val_accuracy: 0.7455 - val_loss: 0.8295
Epoch 7/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.8703 - loss: 0.3829 - val_accuracy: 0.7295 - val_loss: 0.9132
Epoch 8/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.8968 - loss: 0.3019

In [ ]:
data_agumentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.2),
    layers.RandomRotation(0.2)
])

In [ ]:
model_2 = tf.keras.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),
    data_agumentation,
    layers.Rescaling(1./ 255),

    layers.Conv2D(16, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPool2D(),
    layers.Dropout(0.3),

    layers.Conv2D(32, kernel_size=(5, 5), padding='same', activation='relu'),
    layers.MaxPool2D(),
    layers.Dropout(0.3),

    layers.Conv2D(64, kernel_size=(5, 5), padding='same', activation='relu'),
    layers.MaxPool2D(),
    layers.Dropout(0.3),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(classes), activation='softmax')
])

model_2.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model_2.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_2 (Sequential)       │ (None, 220, 220, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_2 (Rescaling)         │ (None, 220, 220, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 220, 220, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 110, 110, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 110, 110, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 110, 110, 32)   │        12,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 55, 55, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 55, 55, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 55, 55, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 27, 27, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 27, 27, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 46656)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │     5,972,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,037,543 (23.03 MB)

 Trainable params: 6,037,543 (23.03 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model_2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 20s 52ms/step - accuracy: 0.3599 - loss: 1.6488 - val_accuracy: 0.5232 - val_loss: 1.3261
Epoch 2/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 14s 51ms/step - accuracy: 0.5482 - loss: 1.2350 - val_accuracy: 0.5955 - val_loss: 1.1383
Epoch 3/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.6050 - loss: 1.0986 - val_accuracy: 0.6692 - val_loss: 0.9569
Epoch 4/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.6420 - loss: 1.0122 - val_accuracy: 0.6634 - val_loss: 0.9536
Epoch 5/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 13s 47ms/step - accuracy: 0.6594 - loss: 0.9574 - val_accuracy: 0.7031 - val_loss: 0.8525
Epoch 6/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.6820 - loss: 0.8981 - val_accuracy: 0.7116 - val_loss: 0.8145
Epoch 7/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 22s 50ms/step - accuracy: 0.6941 - loss: 0.8618 - val_accuracy: 0.7317 - val_loss: 0.7944
Epoch 8/100
280/280 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.7060 - loss: 0

In [ ]:
model_2.save("cnn_accuracy_84.keras")